In [31]:
!pip install sentence-transformers chromadb
# 문장을 벡터(임베딩)로 바꾸는데 사용되는 모델 라이브러리
# 로컬에서 직접 임베딩 할때만 필요

In [32]:
!pip show chromadb

Name: chromadb
Version: 1.3.4
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, posthog, pybase64, pydantic, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 


In [33]:
import chromadb
from chromadb import PersistentClient    # DB 접속자 역할
print(chromadb.__file__)    # 경로 확인

# chroma는 DuckDB 기반의 벡터 저장소로 작동한다
client = PersistentClient(path=".chroma")
!pwd
!ls -a


/usr/local/lib/python3.12/dist-packages/chromadb/__init__.py
/content
.  ..  .chroma	.config  sample_data


In [38]:
# Collection : RDMS의 Table과 유사한 개념이다
collection = client.get_or_create_collection("test")    # 기본 내장 모델 : all-MiniLM-L6-v2 -> 384차원으로 만들어줌, 다른 모델을 지정할 수 있다
print(collection, " ", collection.id)

# 문서를 벡터화해서 DB에 저장
texts = ["Hello World", "Hello Chroma"]
ids = ["doc1", "doc2"]
metas = [{"source":"greeting"}, {"source":"statement"}]

embedding_fn = collection._embedding_function    # 임베딩 함수 반환 (텍스트 -> 벡터화 수행)
embeddings = embedding_fn(texts)
print(type(embeddings), len(embeddings), len(embeddings[0]))    # <class 'list'> 2 384

for i, vector in enumerate(embeddings):
  print(f"문서 : {texts[i]}")
  print(f"임베딩 벡터 앞 5개만 출력 : {vector[:5]}")
  print(f"차원 수 : {len(vector)}")
  print("---------------------")
print("======================\n")

# 두 문장 간 코사인 유사도 확안
from sklearn.metrics.pairwise import cosine_similarity
sin = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
print(f'코사인 유사도 : {sin:.5f}')    # 1에 근사할 수록 유사하다/관련이 깊다. 90도 이면 아예 관련이 없다. -1이면 반대 개념

Collection(name=test)   1b1a57d3-4a01-4404-b0fe-36b10adff4fc
<class 'list'> 2 384
문서 : Hello World
임베딩 벡터 앞 5개만 출력 : [-0.03447731  0.03102319  0.00673492  0.02610894 -0.03936205]
차원 수 : 384
---------------------
문서 : Hello Chroma
임베딩 벡터 앞 5개만 출력 : [-0.12020152  0.03339408  0.00559468  0.02555901 -0.09892537]
차원 수 : 384
---------------------

코사인 유사도 : 0.46819


In [39]:
# collection에 문서 + 벡터 + 메타데이터 저장
collection.add(    # 저장
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=metas
    # urls=['https://example.com/test1/']
)

print(collection)

# collection에 저장된 자료 조회
results = collection.get(include=['documents', 'metadatas'])
print(results)

for doc, meta, id in zip(results['documents'], results['metadatas'], results['ids']):
  print(f"ids : {id}")
  print(f"doc : {doc}")
  print(f"meta : {meta}")
  print("----------------")

print("저장된 문서 id 목록 : ", collection.get()['ids'])

results_vec = collection.get(include=['embeddings'])

# 첫번째 문서의 임베딩 백터 자료 출력
first_embedding = results_vec['embeddings'][0]
print("임베딩 차원수 :", first_embedding[:2], '', len(first_embedding))

print()
for id_, embedding in zip(results['ids'], results_vec['embeddings']):
  print(f"id : {id_}")
  print(f"임베딩 앞 5개 : {embedding[:5]}")

Collection(name=test)
{'ids': ['doc1', 'doc2'], 'embeddings': None, 'documents': ['Hello World', 'Hello Earth'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'source': 'greeting'}, {'source': 'statement'}]}
ids : doc1
doc : Hello World
meta : {'source': 'greeting'}
----------------
ids : doc2
doc : Hello Earth
meta : {'source': 'statement'}
----------------
저장된 문서 id 목록 :  ['doc1', 'doc2']
임베딩 차원수 : [-0.03447731  0.03102319]  384

id : doc1
임베딩 앞 5개 : [-0.03447731  0.03102319  0.00673492  0.02610894 -0.03936205]
id : doc2
임베딩 앞 5개 : [ 0.00198162  0.02486334  0.08626754 -0.0013975  -0.01592566]


In [36]:
# 벡터 기반 유사도
query_text = "Chroma에 대해 설명해줘"    # 검색용 질문
query_embedding = embedding_fn(query_text)[0]    # 문장을 벡터화

# Chroma에 저장된 자료 중에서 유사 자료 검색
search_result = collection.query(
    query_embeddings=[query_embedding],    # 질문을 벡터로 바꾼 결과로 대입
    n_results=2,    # 유사도가 높은 자료 2개 반환
    include=['documents', 'metadatas', 'distances']

)

print(search_result)    # 검색 결과 출력

# 결과 출력
for i, (doc, meta, dist) in enumerate(zip(
    search_result['documents'][0],
    search_result['metadatas'][0],
    search_result['distances'][0])):
  print(f"\n결과 {i+1}")
  print(f" document : {doc}")
  print(f" metadata : {meta}")
  print(f" distance(유사도 거리) : {dist:.4f}")

{'ids': [['doc2', 'doc1']], 'embeddings': None, 'documents': [['Hello Earth', 'Hello World']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'source': 'statement'}, {'source': 'greeting'}]], 'distances': [[1.621009111404419, 1.6471598148345947]]}

결과 1
 document : Hello Earth
 metadata : {'source': 'statement'}
 distance(유사도 거리) : 1.6210

결과 2
 document : Hello World
 metadata : {'source': 'greeting'}
 distance(유사도 거리) : 1.6472
